Making simple attention mechanism showing attention weights from a string of text vectors

In [1]:
import torch
import os
import sys
import re
sys.path.append(os.path.expanduser("~/LLM/tokenizer"))
from SimpleTokenizer import SimpleTokenizer

In [2]:
input_words = "hello, we are trying to use attention"

with open("/home/vivaswan/LLM/tokenizer/sample.txt",'r') as f:
    raw_text = f.read()

processed = re.split(r'[.,/?;: \s]',raw_text)
processed = [word.strip() for word in processed if word.strip()]
alll  = sorted(set(processed))


In [3]:
vocab = {word:id for id,word in enumerate(alll)}
vocab_size = len(alll)
print(vocab_size)

1268


In [4]:
tokenizer = SimpleTokenizer(vocab)

In [5]:
tok = SimpleTokenizer(vocab)
tok_input = tok.encode(input_words)
print(tok_input) ## size is three because there are no unk tokens in the vocab since I used the SimpleTokenizer







[246, 1142, 261]


In [6]:
## now that we have word tokens we use embedding library to convert the ttokens into word embeddings
import torch.nn as nn
from torch.nn import Embedding
from torch.nn import Module

embedding_layer  = nn.Embedding(num_embeddings=vocab_size,embedding_dim=3) ## dims as taken in book


In [7]:
embedded_input = embedding_layer(torch.tensor(tok_input))
print(embedded_input)

tensor([[ 1.4998, -1.1968,  1.4193],
        [-1.9800,  0.5994,  0.9942],
        [ 0.4368, -1.5380, -0.0802]], grad_fn=<EmbeddingBackward0>)


In [8]:
attention_weights=torch.zeros(3,3) ## should be a 3x3 matrix as for each word there are going to be three attention weights where aij represents the weight for jth word in the context vector for i

print(attention_weights)
n = embedded_input.size(0)

context_vectors = torch.zeros(3,3)

for i in range (n):
    for j in range (n):
        attention_weights[i][j] = torch.matmul(embedded_input[i],embedded_input[j].T)

print(attention_weights[0])




tensor([[0., 0., 0.],
        [0., 0., 0.],
        [0., 0., 0.]])
tensor([ 5.6963, -2.2759,  2.3821], grad_fn=<SelectBackward0>)


/tmp/ipykernel_28958/896749398.py:10: UserWarning: The use of `x.T` on tensors of dimension other than 2 to reverse their shape is deprecated and it will throw an error in a future release. Consider `x.mT` to transpose batches of matrices or `x.permute(*torch.arange(x.ndim - 1, -1, -1))` to reverse the dimensions of a tensor. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4481.)
  attention_weights[i][j] = torch.matmul(embedded_input[i],embedded_input[j].T)


In [9]:
## doing normalisation on attention weights
sizee = attention_weights.shape
average_attn_weights = torch.zeros(sizee)
for i in range(n):
    average_attn_weights[i] = attention_weights[i]/torch.sum(attention_weights[i])
print(average_attn_weights)
avg_softmax = torch.softmax(attention_weights,dim=1)
print(avg_softmax)
print(avg_softmax[i].shape)

tensor([[ 0.9817, -0.3922,  0.4105],
        [-2.0216,  4.6796, -1.6579],
        [ 0.7738, -0.6063,  0.8325]], grad_fn=<CopySlices>)
tensor([[9.6459e-01, 3.3271e-04, 3.5077e-02],
        [5.2857e-04, 9.9868e-01, 7.9601e-04],
        [4.5203e-01, 6.4570e-03, 5.4151e-01]], grad_fn=<SoftmaxBackward0>)
torch.Size([3])


In [10]:
for i in range (n):
    context_vectors[i] = torch.matmul(avg_softmax[i],embedded_input) ## the attention weights are designed to have for each embedded input a row of attention weights of size of the input, wheere each column gies a weight that we have to mulitply to the designated embedded vector and then these weighted vectors get added top give you the context vecctpr pf 1 embedded input

print(context_vectors)

tensor([[ 1.4614, -1.2082,  1.3666],
        [-1.9762,  0.5967,  0.9936],
        [ 0.9017, -1.3700,  0.6046]], grad_fn=<CopySlices>)


Now making key,value and query matrices which are essentially linear transformations of the embedded vector matrices . Includes part 3.4.1 of the book

In [11]:
## we are going to be using torch.nn.parameters for the transformation matrices because it keeps and marks these weights as trainable parameters
d_in = 3
d_out = 2

W_q = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad = False) 
W_k = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad = False)
W_v = torch.nn.Parameter(torch.rand(d_in,d_out),requires_grad = False) # to determine the size of the weight matrix we do simple linear algebra. we are multiplying the embedded input of size 1xd_in with weights whose rows must be d_in for mat mul. we want output to have d_out ie 2 size so we set the weights as d_inxd_out

In [12]:
print(W_q)

Parameter containing:
tensor([[0.7227, 0.4990],
        [0.9311, 0.5776],
        [0.0798, 0.4134]])


In [13]:
all_query_vectors=torch.matmul(embedded_input,W_q)
all_key_vectors = torch.matmul(embedded_input,W_k)
all_value_vectors = torch.matmul(embedded_input,W_v)
print(all_query_vectors)

tensor([[ 0.0827,  0.6438],
        [-0.7934, -0.2308],
        [-1.1228, -0.7036]], grad_fn=<MmBackward0>)


In [14]:
## onw we want for all queries ie running a loop through len(queries) , getting a creating of the attention weights which is a matmul of that query with the entire set of key vectors(this is our dot product) and our subsequent multiplication ofthe weights
trainable_attention_weights = torch.matmul(all_query_vectors,(all_key_vectors.T)) ## all_q_v has size(nxd_out) and all_k_v has size (n x d_out), we take key ka transpose and then do matrix multiplication to get nxn matric of weights. notice that for context vector of each query, weights for each key lies along the columns of that query's row number
#print(trainable_attention_weights.shape)
trainable_attention_weights = trainable_attention_weights/(d_out**0.5)
trainable_attention_weights = torch.softmax(trainable_attention_weights,dim = 1)
task_specific_context_vectors = torch.matmul(trainable_attention_weights,all_value_vectors)
print(task_specific_context_vectors)

tensor([[ 0.5085,  0.0590],
        [-0.1585, -0.2137],
        [-0.3105, -0.2809]], grad_fn=<MmBackward0>)


3.4.2 Implementation

In [15]:
class self_attention_v1(nn.Module):
    def __init__(self,din,dout):
        super().__init__()
        self.Wq = nn.Parameter(torch.rand(din,dout))
        self.Wk = nn.Parameter(torch.rand(din,dout))
        self.Wv = nn.Parameter(torch.rand(din,dout))
    def forward(self,embedded_input):
        query_vectors = torch.matmul(embedded_input,self.Wq)
        keys = torch.matmul(embedded_input,self.Wk)
        values = torch.matmul(embedded_input,self.Wv)
        attention_weights = torch.matmul(keys,query_vectors.T)
        attention_weights = torch.softmax(attention_weights/keys.shape[-1]**5,dim = 1)
        context_v = torch.matmul(attention_weights,values)
        return context_v
    def weights(self):
        attention_weights = torch.matmul(self.Wk,self.Wq.T)
        return attention_weights


attn = self_attention_v1(d_in,d_out)
context_vectors = attn.forward(embedded_input)
print(context_vectors)

tensor([[0.2594, 0.1976],
        [0.1500, 0.0843],
        [0.1774, 0.1159]], grad_fn=<MmBackward0>)


implementation of 3.5.1,3.5.2,3.5.3

In [16]:
weights_for_masking = attn.weights()
dimen = weights_for_masking.shape
print(weights_for_masking)

for i  in range (weights_for_masking.shape[0]):
    for j in range(i+1,weights_for_masking.shape[1]):
        weights_for_masking[i][j] = -torch.inf
print(weights_for_masking)

tensor([[0.6220, 0.8393, 1.2987],
        [0.1073, 0.1407, 0.2357],
        [0.5691, 0.7083, 1.3607]], grad_fn=<MmBackward0>)
tensor([[0.6220,   -inf,   -inf],
        [0.1073, 0.1407,   -inf],
        [0.5691, 0.7083, 1.3607]], grad_fn=<CopySlices>)


In [17]:
new_weights = torch.softmax(weights_for_masking,dim = 1)
print(new_weights)

tensor([[1.0000, 0.0000, 0.0000],
        [0.4916, 0.5084, 0.0000],
        [0.2296, 0.2638, 0.5066]], grad_fn=<SoftmaxBackward0>)


In [18]:
## using dropout

dropout = nn.Dropout(0.5) ## the input dropout classes take  are probabilities for dropping /masking variables
droppeed_attention = dropout(new_weights)

In [19]:
print(droppeed_attention)

tensor([[2.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000],
        [0.0000, 0.5276, 0.0000]], grad_fn=<MulBackward0>)


In [22]:
class casual_attention(nn.Module):
    def __init__(self,d_in,d_out,dropout):
        self.Wq = nn.Parameter(d_in,d_out)
        self.Wk = nn.Parameter(d_in,d_out)
        self.Wv = nn.Parameter(d_in,d_out)
        self.dropout - nn.Dropout(dropout)

    def forward(self,embedded_input):
        queries = torch.matmul(embedded_input,self.Wq)
        keys = torch.matmul(embedded_input,self.Wk)
        values = torch.matmul(embedded_input,self.Wv)
        attention_weights = torch.matmul(queries,keys.T)
        for i in range(attention_weights.shape[0]):
            for j in range(i+1,attention_weights.shape[1]):
                attention_weights[i][j] = -torch.inf
        attention_weights = nn.Softmax(attention_weights/keys.shape[-1]**0.5,dim =1)
        attention_weights = dropout(attention_weights)
        context_vec = torch.matmul(values,attention_weights)
        return context_vec

        